# Credit Risk Classification and K-Means — Reproducible Notebook
Run in Google Colab. Upload `german_credit_data.csv` when prompted.

In [ ]:
import pandas as pd, numpy as np
from google.colab import files
uploaded = files.upload()
filename = next(iter(uploaded))
df = pd.read_csv(filename)
df.head()

In [ ]:
df = df.drop(columns=[c for c in df.columns if str(c).lower().startswith('unnamed')], errors='ignore')
for c in ['Saving accounts', 'Checking account']:
    df[c] = df[c].fillna('Unknown')
df = df.drop_duplicates().reset_index(drop=True)
print(df.shape)
print(df.isna().sum())
print(df['Risk'].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
X = df.drop(columns=['Risk'])
y = df['Risk'].map({'good': 0, 'bad': 1})
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()
pre = ColumnTransformer([('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), num_cols), ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)])
model = Pipeline([('prep', pre), ('clf', LogisticRegression(max_iter=1500, random_state=42))])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model.fit(X_train, y_train)
pred = model.predict(X_test); prob = model.predict_proba(X_test)[:, 1]
print('Accuracy:', accuracy_score(y_test, pred))
print('Bad-risk precision:', precision_score(y_test, pred))
print('Bad-risk recall:', recall_score(y_test, pred))
print('Bad-risk F1:', f1_score(y_test, pred))
print('ROC-AUC:', roc_auc_score(y_test, prob))
print('Confusion matrix:\n', confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, target_names=['good','bad']))

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
features = ['Age', 'Credit amount', 'Duration']
Z = StandardScaler().fit_transform(df[features])
km = KMeans(n_clusters=2, random_state=42, n_init=10)
labels = km.fit_predict(Z)
print('Silhouette score:', silhouette_score(Z, labels))
df['Customer Segment'] = labels + 1
print(df.groupby('Customer Segment').agg(Customers=('Risk','size'), Average_Age=('Age','mean'), Average_Credit_Amount=('Credit amount','mean'), Average_Duration=('Duration','mean'), Good=('Risk',lambda s:(s=='good').sum()), Bad=('Risk',lambda s:(s=='bad').sum())))
df.to_csv('credit_risk_cleaned_with_segments.csv', index=False)
files.download('credit_risk_cleaned_with_segments.csv')